In [137]:
import os
import time
import pandas as pd
from pathlib import Path

from datetime import datetime
from dateutil.relativedelta import relativedelta
from dotenv import load_dotenv
from openaq import OpenAQ
from pandas import json_normalize

load_dotenv(override=True)

True

In [156]:
API_KEY = os.getenv("OPENAQ_API_KEY")
SAVE_TO_PATH = Path("../data/data.parquet")
LATITUDE = float(os.getenv("LATITUDE"))
LONGITUDE = float(os.getenv("LONGITUDE"))
RADIUS = 5_00
LIMIT = 20

In [145]:
client = OpenAQ(api_key=API_KEY)

In [146]:
locs = client.locations.list(
    coordinates=(LATITUDE, LONGITUDE),
    radius=RADIUS,
    limit=LIMIT
)

In [150]:
id = None
years = [2025, 2026]
months = [i + 1 for i in range(0, 12)]
sensor_name = "pm25"

for res in locs.results:
    for sensor in res.sensors:
        if sensor.name.startswith("pm25"):
            id = sensor.id
            break

df_list = []
for year in years:
    for month in months:
        date_from = datetime(year, month, 1)
        date_to = date_from + relativedelta(months=1)
        if (year, month) == (2026, 3):
            break
        measurements = client.measurements.list(
            sensors_id=id,
            data="hours",
            datetime_from=date_from,
            datetime_to=date_to
        )
        df_temp = json_normalize(measurements.dict()["results"])
        df_list.append(df_temp)

df_final = pd.concat(df_list).reset_index()

In [157]:
df_final.to_parquet(SAVE_TO_PATH, index=False)

In [161]:
df_final.describe()

,index,value,parameter.id,summary.min,summary.q02,summary.q25,summary.median,summary.q75,summary.q98,summary.max,summary.avg,coverage.expected_count,coverage.observed_count,coverage.percent_complete,coverage.percent_coverage
count,9089.000000,9089.000000,9089.0,9089.000000,9089.000000,9089.000000,9089.000000,9089.000000,9089.000000,9089.000000,9089.000000,9089.0,9089.0,9089.0,9089.0
mean,325.475520,11.880190,2.0,11.877783,11.877783,11.877783,11.877783,11.877783,11.877783,11.877783,11.877783,1.0,1.0,100.0,100.0
std,189.770181,14.216268,0.0,14.216355,14.216355,14.216355,14.216355,14.216355,14.216355,14.216355,14.216355,0.0,0.0,0.0,0.0
min,0.000000,0.250000,2.0,0.250000,0.250000,0.250000,0.250000,0.250000,0.250000,0.250000,0.250000,1.0,1.0,100.0,100.0
25%,162.000000,6.080000,2.0,6.080000,6.080000,6.080000,6.080000,6.080000,6.080000,6.080000,6.080000,1.0,1.0,100.0,100.0
50%,324.000000,9.560000,2.0,9.560000,9.560000,9.560000,9.560000,9.560000,9.560000,9.560000,9.560000,1.0,1.0,100.0,100.0
75%,486.000000,14.500000,2.0,14.530000,14.530000,14.530000,14.530000,14.530000,14.530000,14.530000,14.530000,1.0,1.0,100.0,100.0
max,712.000000,616.000000,2.0,616.280000,616.280000,616.280000,616.280000,616.280000,616.280000,616.280000,616.280000,1.0,1.0,100.0,100.0
